# 08 — Reproduce the release and transfer the procedure

**You will:** reproduce a frozen selection with `pleb dataset-build`, verify every
file, test a changed-input failure, and distinguish checkpoint replay from raw-data analysis.
Keep all observed PTA data outside the PLEB source repository.

In [ ]:
from pathlib import Path
import os, sys, json, shutil, subprocess, hashlib, tarfile
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tomlkit
COURSE = Path(os.environ['PLEB_COURSE_HOME'])
sys.path.insert(0, str(COURSE))
import release_tools as lab
ROOT = lab.environment(Path(os.environ['PLEB_LEARNING_OUTPUT']))
REPO = ROOT/'dataset-repo'
CONFIGS = ROOT/'configs'
PULSARS = lab.PULSARS
CLI = [sys.executable, '-m', 'pleb.cli']

In [ ]:
lab.command(['git','-C',REPO,'log','--all','--graph','--decorate','--oneline'])
display(pd.DataFrame([
    ('01_ingest.toml','scaffold','raw_ingest','mapping and approved raw inputs'),
    ('02_prepare.toml','raw_ingest','prepared','flags, duplicates and variants'),
    ('03_detect.toml','prepared','prepared','diagnostics only'),
    ('05_apply.toml','prepared','reviewed','preserved reviewed QC CSVs'),
    ('06_final_fit.toml','reviewed','reviewed','fresh timing/QC diagnostics'),
    ('explicit model commit','reviewed','release_model','approved fitted PARs'),
],columns=['operation','input_branch','output_branch','decision/input']))

## Native reconstruction from a reviewed checkpoint

A checkpoint recipe does not repeat ingest, detection, human review or fitting.
It freezes the input commit, reviewed transformation catalogs, expected output files
and PLEB code hashes. One PLEB command then verifies and publishes the selection.
The earlier lessons teach how to obtain that checkpoint; they are still necessary.

The native recipe currently counts `*_combined_all.tim`. Build a training checkpoint
with combined aliases of the **approved canonical pair**, removing unvalidated subset
variants. No measurements or coefficients are changed here. The output branch is local;
PLEB's private-index publication leaves the training checkout unchanged.

In [ ]:
checkout_before = lab.command(['git','-C',REPO,'rev-parse','HEAD']).strip()
checkpoint = ROOT/'checkpoint-worktree'
lab.command(['git','-C',REPO,'worktree','add','-b','release_checkpoint',checkpoint,'release_model'])
for psr in PULSARS:
    folder = checkpoint/'data'/psr
    for variant in ['legacy','new','combined']:
        for suffix in ['.par','_all.tim']:
            (folder/f'{psr}_{variant}{suffix}').unlink(missing_ok=True)
    for suffix in ['.par','_all.tim']:
        shutil.copy2(folder/f'{psr}{suffix}',folder/f'{psr}_combined{suffix}')
lab.command(['git','-C',checkpoint,'add','data'])
lab.command(['git','-C',checkpoint,'commit','-m','Freeze approved synthetic combined pair for native replay'])
lab.command(['git','-C',REPO,'worktree','remove',checkpoint])
commit = lab.command(['git','-C',REPO,'rev-parse','release_checkpoint']).strip()
reference = lab.snapshot(REPO,commit,ROOT/'exports/08_checkpoint')
from pleb.dataset_recipe import active_count, RecipeError, build_recipe
total = sum(active_count(p) for p in reference.glob('J*/*_combined_all.tim'))
assert total==576-len(json.loads((ROOT/'review_approval.json').read_text())['selected_bad'])

## Lock the recipe inputs

This checkpoint already contains the reviewed selection, so the three transformation
catalogs are intentionally empty. On a real restoration use evidence-backed catalogs:
`restoration.csv` supplies missing rows with original TIME context; replacements declare
the approved provider version and system/JUMP changes; keep decisions name a physical
commented row and its hash. The current keep stage requires `-pqc good` and
`-manual_reason keep`; it is not a generic manual-rejection importer.

Create expected hashes from the independently reviewed reference, **before** running
the recipe. Never regenerate the reference from a failed output just to make it pass.
Additional manual bad decisions must first go through the reviewed FixDataset stage
(04–05), then be frozen into a new checkpoint and manifest. Dataset-build does not
ingest arbitrary review UI overrides directly.

In [ ]:
recipe_dir = CONFIGS/'replay'
recipe_dir.mkdir()
(recipe_dir/'restoration.csv').write_text('pulsar,tim,before_line,original_TIME_seconds,restored_row\n')
for name in ['replacements.json','keep_decisions.json']:
    (recipe_dir/name).write_text('[]\n')
(recipe_dir/'system_rules.json').write_text('{}\n')
reference_hashes = lab.hashes(reference)
(recipe_dir/'expected_manifest.json').write_text(json.dumps(reference_hashes,indent=2)+'\n')
import pleb
package = Path(pleb.__file__).parent
profile = dict(recipe_version=1,
    source=dict(repository='../../dataset-repo',commit=commit,dataset='data'),
    stages=dict(restoration='restoration.csv',replacements='replacements.json',
                keep_decisions='keep_decisions.json',system_rules='system_rules.json',
                expected_manifest='expected_manifest.json'),
    files=lab.hashes(recipe_dir),
    code={name:hashlib.sha256((package/name).read_bytes()).hexdigest() for name in
          ['dataset_recipe.py','cli.py','tim_utils.py','system_flag_inference.py']},
    expected=dict(baseline=total,restored=0,reactivated=0,old_backend=0,new_backend=0,
                  changed_parfiles=0,active=total),
    output=dict(dataset='data',config_directory='configs/replay',branch='release_replayed'))
recipe_path = lab.write_config(recipe_dir/'recipe.toml',profile)
lab.command(CLI+['dataset-build','--config',recipe_path,'--output-dir',ROOT/'replay-1'],
            log=ROOT/'logs/08_replay.log')
summary = json.loads((ROOT/'replay-1/summary.json').read_text())
assert summary['selection_verified'] and not summary['fitting_rerun'] and not summary['qc_rerun']
replayed = ROOT/'replay-1/dataset'
actual = lab.hashes(replayed)
generated = actual.pop('build_provenance/recipe_verification.json')
assert actual==reference_hashes
assert lab.command(['git','-C',REPO,'rev-parse','HEAD']).strip()==checkout_before
display(pd.DataFrame([summary['counts']]))

In [ ]:
# A second native run verifies idempotent publication of the same local branch.
lab.command(CLI+['dataset-build','--config',recipe_path,'--output-dir',ROOT/'replay-2'],
            log=ROOT/'logs/08_replay_repeat.log')
assert lab.hashes(ROOT/'replay-2/dataset')==lab.hashes(replayed)
# Deliberately break a COPY of a locked input. Fail before changing the dataset.
broken = ROOT/'tampered-recipe'
shutil.copytree(recipe_dir,broken)
(broken/'keep_decisions.json').write_text('[{}]\n')
try:
    build_recipe(broken/'recipe.toml',ROOT/'must-not-build',source_repo=REPO)
except RecipeError as error:
    assert 'Locked input changed' in str(error)
    print('Expected rejection:',error)
else:
    raise AssertionError('Changed review input was silently accepted')
assert not (ROOT/'must-not-build').exists()
(ROOT/'replay_recipe.json').write_text(json.dumps(dict(commit=summary['output_commit'],
    active=total,profile=str(recipe_path)),indent=2))

## Independent recreation exercise

1. Launch `./learn.sh release /another/output --port 8890` in a separate terminal.
2. Run notebooks 00–03. Compare the raw manifests and prepared **TOA identities/states**,
   not timestamped Git commits or log bytes.
3. Transfer the reviewed QC tables and recorded approval from the first run as explicit
   inputs, verify their hashes, then run 05. This tests replay of preserved decisions.
   Repeating notebook 04 as a fresh human review is a different experiment.
4. Fit and review the model again, then freeze a new archive. Compare the active/commented
   selection and residual differences within a declared numerical tolerance.

The configs use absolute container-visible paths. Adapt `home_dir`, `results_dir`, mapping
roots and reviewed-QC paths for the new workspace. Do not silently point the second run
back at the first dataset. Deterministic selection and byte-identical output archives are
different promises: fit formatting, library versions, timestamps and provenance can differ.

After separate stages are understood, PLEB's `workflow` mode can chain known configs.
It does not supply a human review. Never put detection and automatic apply into a workflow
as a substitute for deciding and preserving your action policy.

In [ ]:
workflow = {
    'config':str(CONFIGS/'02_prepare.toml'),'mode':'serial',
    'groups':[
        {'name':'prepare','mode':'serial','steps':[{'name':'pipeline','config':str(CONFIGS/'02_prepare.toml')}]},
        {'name':'detect','mode':'serial','steps':[{'name':'pipeline','config':str(CONFIGS/'03_detect.toml')}]},
    ]}
path = lab.write_config(CONFIGS/'prepare_then_detect.toml',workflow)
print('For a FRESH workspace after ingest:')
print('pleb workflow --file',path)
print('Do not run this on the current workspace: prepared already exists.')

## Replace synthetic inputs deliberately

| Training choice | What a real PTA release must supply |
|---|---|
| Two generated pulsars | Full expected pulsar/backend/file inventory, permissions and input versions |
| Known truth PARs | Independently approved starting models and model-change review |
| Artificial zero-clock station | Telescope observatory records, clocks and adopted time/ephemeris conventions |
| A/B flags and exact duplicates | Backend identities, reference systems, overlap preferences and matching tolerances |
| Isolated white-noise outliers | Validated detector settings, event/eclipse/solar policies and backend thresholds |
| Training review IDs | Stable TOA identity, keep/comment reason, reviewer, source table and hashes |
| One full-TOA fit per pulsar | Fits and residual checks for each published PAR/TIM variant, plus noise assumptions |
| A labelled teaching archive | Release version, provenance, licence/access rules, validation and transfer checksum |

Start with one real pulsar in a new private workspace, inspect each hand-off, then add
another pulsar before scaling to the full PTA. Use the production playbook for the
project's agreed expert checkpoints. This training recipe is not an EPTA historical replay.

**Your final task:** write a runlist that names each config, input branch, output branch,
report path, reviewer checkpoint and acceptance condition. Have another person follow
it from the preserved inputs without relying on your shell history.